# Group J - Problem Set 2

## Queueing Theory Simulations

In [1]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import random
import math
import pandas as pd

## a) Arrival and Service Rate Estimates

In [2]:
# data
arrival_nums = np.array([12, 12, 13]) * 1/5
service_nums = np.array([15, 10, 17]) * 1/5

arrival_rate_est = sum(arrival_nums)/len(arrival_nums)
service_rate_est = sum(service_nums)/len(service_nums)

print(f"Average Arrival Rate (per minute): {round(arrival_rate_est, 3)}")
print(f"Average Service Rate (per minute): {round(service_rate_est, 3)}")

Average Arrival Rate (per minute): 2.467
Average Service Rate (per minute): 2.8


## b) M/M/1 Queue Simulation

In [3]:
# inverse-transform sampling for random numbers:
def exponential_sample(lamda):
    r = random.random()
    return -math.log(r) / lamda

def generate_samples(d):
    samples_interarrival = [exponential_sample(arrival_rate_est) for _ in range(d)]
    samples_service = [exponential_sample(service_rate_est) for _ in range(d)]
    return samples_interarrival, samples_service

In [4]:
# set up recursion function using Lindley's equation
def recurse_lindley(service_times, interarrival_times, d):
    
    waiting_times = [0]*d
    for i in range(d-1):
        waiting_times[i+1] = max(0, waiting_times[i] + service_times[i] - interarrival_times[i])
    time_in_system = [waiting_times[i] + service_times[i] for i in range(d)]

    avg_wait_queue = sum(waiting_times) / d
    avg_system_time = sum(time_in_system) / d
    return waiting_times

In [5]:
# record events (when customers enter or leave)
def record_events(service_times, interarrival_times, waiting_times, d):

    # generate events tracker
    arrival_times = [0]*d
    for i in range(1,d):
        arrival_times[i] = arrival_times[i-1] + interarrival_times[i-1]
    service_start_times = [arrival_times[i] + waiting_times[i] for i in range(d)]
    service_end_times = [service_start_times[i] + service_times[i] for i in range(d)]
    events = []
    for time in arrival_times:
        events.append((time, +1))
    for time in service_end_times:
        events.append((time, -1))
    events.sort()

    return events

# report required estimates from simulation run
def report_estimates(waiting_times, events, d):

    # i) estimate average queue length + iii) utilisation ratio
    time_server_occupied = 0
    previous_time = 0
    time_weighted_total_L = 0
    current_L = 0
    for time, change_L in events:
        time_delta = time - previous_time
        if current_L > 0:
            time_server_occupied += time_delta
        time_weighted_total_L += current_L * time_delta
        previous_time = time
        current_L += change_L
    average_L = time_weighted_total_L / previous_time
    utilisation_ratio = time_server_occupied / previous_time
    
    # ii) estimate average wait time
    average_wait = sum(waiting_times) / d

    return np.array([average_L, average_wait, utilisation_ratio])
    

In [6]:
# calculate theoretical values:
rho = arrival_rate_est / service_rate_est
expected_L = rho / (1 - rho)
expected_wait = rho / (service_rate_est - arrival_rate_est)

In [7]:
n = 100
d = 100000

estimates_list = []

for run in range(n):
    random.seed(run)
    samples_interarrival, samples_service = generate_samples(d)
    waiting_times = recurse_lindley(samples_service, samples_interarrival, d)
    events = record_events(samples_service, samples_interarrival, waiting_times, d)
    estimates = report_estimates(waiting_times, events, d)
    estimates_list.append(estimates)

estimates_array = np.array(estimates_list)
means = estimates_array.mean(axis=0)
stds = estimates_array.std(axis=0, ddof=1)
ci95 = 1.96 * stds / np.sqrt(n)

estimate_L, estimate_wait, estimate_utilisation_ratio = means
ci_L, ci_wait, ci_util = ci95
rho = arrival_rate_est / service_rate_est
expected_L = rho / (1 - rho)
expected_wait = rho / (service_rate_est - arrival_rate_est)

data = {
    "Metric": ["Queue Length (L)", "Waiting Time (mins)", "Utilisation Ratio"],
    "Theoretical": [expected_L, expected_wait, rho],
    "Monte Carlo Estimate": means,
    "95% CI (±)": ci95,
    "% Error": np.abs(means - [expected_L, expected_wait, rho]) / [expected_L, expected_wait, rho] * 100,
    "Coverage": [
        "Yes" if abs(means[0] - expected_L) <= ci_L else "No",
        "Yes" if abs(means[1] - expected_wait) <= ci_wait else "No",
        "Yes" if abs(means[2] - rho) <= ci_util else "No"
    ]
}

df = pd.DataFrame(data)
df[["Theoretical", "Monte Carlo Estimate"]] = df[["Theoretical", "Monte Carlo Estimate"]].round(4)
df["95% CI (±)"] = df["95% CI (±)"].round(4)
df["% Error"] = df["% Error"].round(3)

print("=== M/M/1 Queue: Theoretical vs Monte Carlo Estimates ===\n")
print(df.to_string(index=False))

=== M/M/1 Queue: Theoretical vs Monte Carlo Estimates ===

             Metric  Theoretical  Monte Carlo Estimate  95% CI (±)  % Error Coverage
   Queue Length (L)       7.4000                7.3546      0.0736    0.614      Yes
Waiting Time (mins)       2.6429                2.6263      0.0289    0.628      Yes
  Utilisation Ratio       0.8810                0.8800      0.0008    0.109       No
